# Silver Layer - Limpeza, Validacao e Integracao

**Camada:** Silver (tratada)
**Entrada:** `workspace.bronze.*`  |  **Saida:** `workspace.silver.*`

O que esta camada faz:
- Corrige tipos (a bronze e toda string): inteiros, decimais (troca virgula por ponto, remove `>`).
- Padroniza `rede`: os agregados usam codigo (0/2/3/5) e as metas usam texto
  ("Publica"/"Municipal"); os microdados de aluno usam `tp_dependencia` (1-4). Cada caso
  e normalizado para um vocabulario unico (total/estadual/municipal/publica/federal/privada).
- Trata textos (trim) e valores ausentes (string vazia vira null).
- **Trata duplicidades e nulos**: remove duplicatas nas chaves de negocio e descarta linhas
  com chave de IDENTIDADE nula; nulos de METRICA sao preservados (sao informativos - aluno
  ausente/sem medicao). Tudo e registrado num relatorio de tratamento.
- Valida qualidade (duplicados, intervalos, nulos) e faz cross-source Base dos Dados x INEP
  (fonte canonica: os microdados oficiais `ts_*`).
- **Integra as bases** (indicador x meta) gerando tabelas prontas para a Gold.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("tech-challenge-silver").getOrCreate()

BRONZE = "workspace.bronze"
SILVER_CATALOG = "workspace"
SILVER_SCHEMA = "silver"
SILVER = f"{SILVER_CATALOG}.{SILVER_SCHEMA}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
print(f"Schema {SILVER} pronto")

In [ ]:
# ----------------- Funcoes auxiliares -----------------

def limpar_texto(coluna):
    return F.trim(F.col(coluna).cast("string"))


def para_inteiro(coluna):
    v = limpar_texto(coluna)
    return F.when(v == "", None).otherwise(v.cast("int"))


def para_decimal(coluna):
    # CSVs vem com virgula decimal e as vezes valores tipo ">95"
    v = F.regexp_replace(limpar_texto(coluna), ",", ".")
    v = F.regexp_replace(v, ">", "")
    v = F.trim(v)
    return F.when(v == "", None).otherwise(v.cast("double"))


def normalizar_rede(coluna):
    # Aceita codigo (0/2/3/5) e texto ("Publica"/"Municipal"...). Remove acentos.
    v = F.lower(limpar_texto(coluna))
    v = F.translate(v, "áàâãéêíóôõúç", "aaaaeeiooouc")
    return (
        F.when(v.isin("0", "total"), "total")
        .when(v.isin("2", "estadual"), "estadual")
        .when(v.isin("3", "municipal"), "municipal")
        .when(v.isin("5", "publica"), "publica")
        .when(v.isin("1", "federal"), "federal")
        .when(v.isin("4", "privada"), "privada")
        .otherwise(v)
    )


def normalizar_dependencia(coluna):
    # ts_aluno.tp_dependencia: 1=federal, 2=estadual, 3=municipal, 4=privada
    v = limpar_texto(coluna)
    return (
        F.when(v == "1", "federal")
        .when(v == "2", "estadual")
        .when(v == "3", "municipal")
        .when(v == "4", "privada")
        .otherwise(None)
    )


def ler_bronze(tabela):
    return spark.table(f"{BRONZE}.{tabela}")


def salvar_silver(df, tabela):
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{SILVER}.{tabela}"))
    print(f"{tabela}: {df.count():,} registros")

## Agregados: indicadores e metas

In [ ]:
NIVEIS = [f"proporcao_aluno_nivel_{i}" for i in range(9)]

indicador_uf = ler_bronze("indicador_alfabetizacao_uf").select(
    para_inteiro("ano").alias("ano"),
    F.upper(limpar_texto("sigla_uf")).alias("sigla_uf"),
    para_inteiro("serie").alias("serie"),
    limpar_texto("rede").alias("rede_codigo"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("media_portugues").alias("media_portugues"),
    *[para_decimal(c).alias(c) for c in NIVEIS],
)

indicador_municipio = ler_bronze("indicador_alfabetizacao_municipio").select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("id_municipio").alias("id_municipio"),
    para_inteiro("serie").alias("serie"),
    limpar_texto("rede").alias("rede_codigo"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("media_portugues").alias("media_portugues"),
    *[para_decimal(c).alias(c) for c in NIVEIS],
)

In [ ]:
METAS = [f"meta_alfabetizacao_{a}" for a in range(2024, 2031)]

def _metas_alias():
    return [para_decimal(m).alias(m.replace("meta_alfabetizacao_", "meta_")) for m in METAS]

meta_brasil = ler_bronze("meta_alfabetizacao_brasil").select(
    para_inteiro("ano").alias("ano"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    *_metas_alias(),
    para_decimal("percentual_participacao").alias("percentual_participacao"),
)

meta_uf = ler_bronze("meta_alfabetizacao_uf").select(
    para_inteiro("ano").alias("ano"),
    F.upper(limpar_texto("sigla_uf")).alias("sigla_uf"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    *_metas_alias(),
    para_decimal("percentual_participacao").alias("percentual_participacao"),
)

meta_municipio = ler_bronze("meta_alfabetizacao_municipio").select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("id_municipio").alias("id_municipio"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    *_metas_alias(),
    limpar_texto("nivel_alfabetizacao").alias("nivel_alfabetizacao"),
    para_decimal("percentual_participacao").alias("percentual_participacao"),
)

## Microdados: aluno, municipio e estado

In [ ]:
ts_aluno = ler_bronze("ts_aluno").select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("co_uf").alias("codigo_uf"),
    F.upper(limpar_texto("sg_uf")).alias("sigla_uf"),
    limpar_texto("co_municipio").alias("id_municipio"),
    limpar_texto("no_municipio").alias("nome_municipio"),
    limpar_texto("id_escola").alias("id_escola"),
    limpar_texto("id_aluno").alias("id_aluno"),
    para_inteiro("tp_serie").alias("serie"),
    limpar_texto("tp_dependencia").alias("dependencia_codigo"),
    normalizar_dependencia("tp_dependencia").alias("dependencia"),
    para_inteiro("in_presenca_lp").alias("presenca"),
    para_inteiro("in_preenchimento_lp").alias("preenchimento"),
    para_inteiro("co_caderno_lp").alias("caderno"),
    para_decimal("vl_proficiencia_lp").alias("proficiencia"),
    para_decimal("vl_peso_aluno_lp").alias("peso_aluno"),
    para_inteiro("in_alfabetizado").alias("alfabetizado"),
)

NIVEIS_LP = [f"pc_aluno_nivel_{i}_lp" for i in range(9)]

ts_municipio = ler_bronze("ts_municipio").select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("co_uf").alias("codigo_uf"),
    F.upper(limpar_texto("sg_uf")).alias("sigla_uf"),
    limpar_texto("co_municipio").alias("id_municipio"),
    limpar_texto("no_municipio").alias("nome_municipio"),
    para_inteiro("tp_serie").alias("serie"),
    limpar_texto("id_tipo_rede").alias("rede_codigo"),
    normalizar_rede("id_tipo_rede").alias("rede"),
    para_decimal("pc_aluno_alfabetizado").alias("pc_aluno_alfabetizado"),
    para_decimal("vl_media_lp").alias("media_portugues"),
    *[para_decimal(c).alias(c.replace("pc_aluno_nivel_", "nivel_").replace("_lp", "")) for c in NIVEIS_LP],
)

ts_estado = ler_bronze("ts_estado").select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("co_uf").alias("codigo_uf"),
    F.upper(limpar_texto("sg_uf")).alias("sigla_uf"),
    para_inteiro("tp_serie").alias("serie"),
    limpar_texto("id_tipo_rede").alias("rede_codigo"),
    normalizar_rede("id_tipo_rede").alias("rede"),
    para_decimal("pc_aluno_alfabetizado").alias("pc_aluno_alfabetizado"),
    para_decimal("vl_media_lp").alias("media_portugues"),
    *[para_decimal(c).alias(c.replace("pc_aluno_nivel_", "nivel_").replace("_lp", "")) for c in NIVEIS_LP],
)

## Tratamento: deduplicacao e chaves de identidade nulas

Aplicamos o tratamento ativo antes de validar/integrar:
- **Deduplicacao**: `dropDuplicates` nas chaves de negocio (1 registro por chave).
- **Chave de identidade nula**: linhas sem `ano`/`id` de identidade sao descartadas.
- **Nulos de metrica**: preservados (nulo e informativo; imputar distorceria o indicador).

Obs.: em `ts_aluno`, a identidade e `(ano, id_aluno)`; `id_municipio` NAO e identidade, entao
os ~628 alunos de 2025 sem municipio (escolas fora do Censo 2025) sao mantidos.

In [ ]:
tratamentos = []

def tratar_tabela(df, chaves_dedup, chaves_identidade, nome):
    """Remove linhas com chave de identidade nula e deduplica pelas chaves de negocio."""
    antes = df.count()
    cond = None
    for c in chaves_identidade:
        cc = F.col(c).isNull()
        cond = cc if cond is None else (cond | cc)
    sem_nulo = df.filter(~cond) if cond is not None else df
    n_apos_nulo = sem_nulo.count()
    tratado = sem_nulo.dropDuplicates(chaves_dedup)
    depois = tratado.count()
    tratamentos.append({
        "tabela": nome,
        "linhas_entrada": antes,
        "removidas_identidade_nula": antes - n_apos_nulo,
        "removidas_duplicadas": n_apos_nulo - depois,
        "linhas_saida": depois,
    })
    return tratado

In [ ]:
indicador_uf = tratar_tabela(indicador_uf, ["ano","sigla_uf","serie","rede"], ["ano","sigla_uf"], "indicador_alfabetizacao_uf")
indicador_municipio = tratar_tabela(indicador_municipio, ["ano","id_municipio","serie","rede"], ["ano","id_municipio"], "indicador_alfabetizacao_municipio")
meta_brasil = tratar_tabela(meta_brasil, ["ano","rede"], ["ano"], "meta_alfabetizacao_brasil")
meta_uf = tratar_tabela(meta_uf, ["ano","sigla_uf","rede"], ["ano","sigla_uf"], "meta_alfabetizacao_uf")
meta_municipio = tratar_tabela(meta_municipio, ["ano","id_municipio","rede"], ["ano","id_municipio"], "meta_alfabetizacao_municipio")
ts_aluno = tratar_tabela(ts_aluno, ["ano","id_aluno"], ["ano","id_aluno"], "ts_aluno")
ts_municipio = tratar_tabela(ts_municipio, ["ano","id_municipio","serie","rede"], ["ano","id_municipio"], "ts_municipio")
ts_estado = tratar_tabela(ts_estado, ["ano","sigla_uf","serie","rede"], ["ano","sigla_uf"], "ts_estado")

for t in tratamentos:
    print(f"{t['tabela']:38} entrada={t['linhas_entrada']:>9,} "
          f"| id_nula=-{t['removidas_identidade_nula']:,} dup=-{t['removidas_duplicadas']:,} "
          f"| saida={t['linhas_saida']:>9,}")

salvar_silver(spark.createDataFrame(tratamentos), "tratamento_silver")

## Validacao de qualidade

Checagens de duplicidade, intervalos e nulos, alem do **cross-source**:
comparamos a taxa da Base dos Dados com o percentual oficial do INEP (ts_*, fonte canonica).

In [ ]:
resultados = []

def val_duplicados(df, chaves, nome):
    total = df.count()
    unicos = df.select(chaves).distinct().count()
    dup = total - unicos
    resultados.append({"tabela": nome, "tipo": "duplicidade",
                       "detalhe": f"{dup} duplicados em {chaves}",
                       "status": "OK" if dup == 0 else "ALERTA"})

def val_intervalo(df, coluna, lo, hi, nome):
    fora = df.filter(F.col(coluna).isNotNull() & ((F.col(coluna) < lo) | (F.col(coluna) > hi))).count()
    resultados.append({"tabela": nome, "tipo": "intervalo",
                       "detalhe": f"{fora} fora de [{lo},{hi}] em {coluna}",
                       "status": "OK" if fora == 0 else "ALERTA"})

def val_nulos(df, colunas, nome):
    cond = None
    for c in colunas:
        cc = F.col(c).isNull()
        cond = cc if cond is None else (cond | cc)
    n = df.filter(cond).count()
    resultados.append({"tabela": nome, "tipo": "nulos_chave",
                       "detalhe": f"{n} linhas com chave nula em {colunas}",
                       "status": "OK" if n == 0 else "ALERTA"})

In [ ]:
# Duplicidade nas chaves de negocio
val_duplicados(indicador_uf, ["ano", "sigla_uf", "serie", "rede"], "indicador_alfabetizacao_uf")
val_duplicados(indicador_municipio, ["ano", "id_municipio", "serie", "rede"], "indicador_alfabetizacao_municipio")
val_duplicados(meta_uf, ["ano", "sigla_uf", "rede"], "meta_alfabetizacao_uf")
val_duplicados(meta_municipio, ["ano", "id_municipio", "rede"], "meta_alfabetizacao_municipio")
val_duplicados(ts_aluno, ["ano", "id_aluno"], "ts_aluno")

# Intervalos
val_intervalo(indicador_municipio, "taxa_alfabetizacao", 0, 100, "indicador_alfabetizacao_municipio")
val_intervalo(ts_aluno, "proficiencia", 0, 1000, "ts_aluno")
val_intervalo(ts_municipio, "pc_aluno_alfabetizado", 0, 100, "ts_municipio")

# Nulos em chaves de identidade
val_nulos(ts_aluno, ["ano", "id_aluno"], "ts_aluno")
val_nulos(indicador_municipio, ["ano", "id_municipio"], "indicador_alfabetizacao_municipio")

for r in resultados:
    print(f"[{r['status']}] {r['tabela']:38} {r['tipo']:14} {r['detalhe']}")

In [ ]:
# Cross-source: Base dos Dados x INEP (mesma chave, tolerancia de 1 ponto percentual)
def cross_source(base_df, base_taxa, ts_df, ts_taxa, chaves, nome):
    j = (base_df.select(*chaves, F.col(base_taxa).alias("taxa_bdd"))
         .join(ts_df.select(*chaves, F.col(ts_taxa).alias("taxa_inep")), chaves, "inner")
         .filter(F.col("taxa_bdd").isNotNull() & F.col("taxa_inep").isNotNull())
         .withColumn("diff", F.abs(F.col("taxa_bdd") - F.col("taxa_inep"))))
    total = j.count()
    diverg = j.filter(F.col("diff") > 1.0).count()
    media = j.select(F.avg("diff")).first()[0] if total else 0.0
    status = "OK" if diverg == 0 else "ALERTA"
    print(f"[{status}] cross-source {nome}: {diverg}/{total} divergem >1pp (diff medio {media:.4f}pp)")
    resultados.append({"tabela": nome, "tipo": "cross_source",
                       "detalhe": f"{diverg}/{total} divergem >1pp",
                       "status": status})

cross_source(indicador_municipio, "taxa_alfabetizacao", ts_municipio, "pc_aluno_alfabetizado",
             ["ano", "id_municipio", "serie", "rede"], "municipio x ts_municipio")
cross_source(indicador_uf, "taxa_alfabetizacao", ts_estado, "pc_aluno_alfabetizado",
             ["ano", "sigla_uf", "serie", "rede"], "uf x ts_estado")

salvar_silver(spark.createDataFrame(resultados), "validacao_silver")

## Integracao das bases

Diferencial desta Silver: unir indicador + meta numa mesma tabela por municipio e por UF,
ja com o gap ate a meta de 2030 e o codigo da UF derivado. Estas tabelas alimentam a Gold.

In [ ]:
# Municipio: indicador + meta (join por ano, id_municipio, rede)
indicador_municipio_integrado = (
    indicador_municipio.alias("i")
    .join(
        meta_municipio.select(
            "ano", "id_municipio", "rede",
            "meta_2024", "meta_2025", "meta_2026", "meta_2027",
            "meta_2028", "meta_2029", "meta_2030", "percentual_participacao",
        ).alias("m"),
        on=["ano", "id_municipio", "rede"], how="left",
    )
    .withColumn("codigo_uf", F.substring(F.col("id_municipio"), 1, 2))
    .withColumn("gap_meta_2030", F.col("meta_2030") - F.col("taxa_alfabetizacao"))
)

# UF: indicador + meta (join por ano, sigla_uf, rede)
indicador_uf_integrado = (
    indicador_uf.alias("i")
    .join(
        meta_uf.select(
            "ano", "sigla_uf", "rede",
            "meta_2024", "meta_2025", "meta_2026", "meta_2027",
            "meta_2028", "meta_2029", "meta_2030", "percentual_participacao",
        ).alias("m"),
        on=["ano", "sigla_uf", "rede"], how="left",
    )
    .withColumn("gap_meta_2030", F.col("meta_2030") - F.col("taxa_alfabetizacao"))
)

print("integrado municipio:", indicador_municipio_integrado.count(), "linhas")
print("integrado uf:", indicador_uf_integrado.count(), "linhas")

## Salvamento das tabelas Silver

In [ ]:
bases = {
    "indicador_alfabetizacao_uf": indicador_uf,
    "indicador_alfabetizacao_municipio": indicador_municipio,
    "meta_alfabetizacao_brasil": meta_brasil,
    "meta_alfabetizacao_uf": meta_uf,
    "meta_alfabetizacao_municipio": meta_municipio,
    "ts_aluno": ts_aluno,
    "ts_municipio": ts_municipio,
    "ts_estado": ts_estado,
    "indicador_municipio_integrado": indicador_municipio_integrado,
    "indicador_uf_integrado": indicador_uf_integrado,
}
for nome, df in bases.items():
    salvar_silver(df, nome)